<a href="https://colab.research.google.com/github/xxcorn888-cyber/solubility-prediction/blob/main/bbb_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 45.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
bbb = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
bbb.head()

,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
bbb["mol"] = bbb["smiles"].apply(Chem.MolFromSmiles)
bbb = bbb[bbb["mol"].notnull()]
bbb["MolWt"] = bbb["mol"].apply(Descriptors.MolWt)
bbb["LogP"] = bbb["mol"].apply(Descriptors.MolLogP)
bbb["TPSA"] = bbb["mol"].apply(Descriptors.TPSA)
bbb["HBD"] = bbb["mol"].apply(Descriptors.NumHDonors)
bbb["HBA"] = bbb["mol"].apply(Descriptors.NumHAcceptors)
bbb[["smiles","MolWt","LogP","TPSA","HBD","HBA"]].head()

[18:51:18] Explicit valence for atom # 1 N, 4, is greater than permitted
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] Explicit valence for atom # 6 N, 4, is greater than permitted
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] Explicit valence for atom # 6 N, 4, is greater than permitted
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] WARNING: not removing hydrogen atom without neighbors
[18:51:18] Explicit valence for atom # 11 N, 4, is greater than pe

,smiles,MolWt,LogP,TPSA,HBD,HBA
0,[Cl].CC(C)NCC(O)COc1cccc2ccccc12,294.802,3.26700,41.49,2,3
1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl,360.325,4.63500,29.54,0,3
2,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...,361.373,1.54400,75.01,1,5
3,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C,290.407,2.57750,41.57,1,3
4,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...,435.889,2.54872,112.74,2,6


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X = bbb[["MolWt","LogP","TPSA","HBD","HBA"]]
y = bbb["p_np"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8357843137254902

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report
print(bbb["p_np"].value_counts(normalize=True))
proba = clf.predict_proba(X_test)[:,1]
print("AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, clf.predict(X_test)))

p_np
1    0.765081
0    0.234919
Name: proportion, dtype: float64
AUC: 0.8480271975417607
              precision    recall  f1-score   support

           0       0.72      0.54      0.61        99
           1       0.86      0.93      0.90       309

    accuracy                           0.84       408
   macro avg       0.79      0.73      0.75       408
weighted avg       0.83      0.84      0.83       408



In [ ]:
clf = RandomForestClassifier(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:,1]
print("AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, clf.predict(X_test)))

AUC: 0.8396587231538688
              precision    recall  f1-score   support

           0       0.70      0.53      0.60        99
           1       0.86      0.93      0.89       309

    accuracy                           0.83       408
   macro avg       0.78      0.73      0.75       408
weighted avg       0.82      0.83      0.82       408



In [ ]:
from rdkit.Chem import AllChem
import numpy as np
fps = [[int(b) for b in AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048).ToBitString()] for m in bbb["mol"]]
X = np.array(fps)
y = bbb["p_np"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:,1]
print("AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, clf.predict(X_test)))

[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerator
[18:59:27] DEPRECATION WARNING: please use MorganGenerat

AUC: 0.8660717204406525
              precision    recall  f1-score   support

           0       0.81      0.47      0.60        99
           1       0.85      0.96      0.90       309

    accuracy                           0.85       408
   macro avg       0.83      0.72      0.75       408
weighted avg       0.84      0.85      0.83       408



In [ ]:
test_smiles = "C1=CC(=CC=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O"
m = Chem.MolFromSmiles(test_smiles)
fp = [int(b) for b in AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048).ToBitString()]
prob = clf.predict_proba([fp])[0][1]
print("能进脑的概率:", prob)

能进脑的概率: 0.49


[19:07:51] DEPRECATION WARNING: please use MorganGenerator


In [ ]:
print("TPSA:", Descriptors.TPSA(m))
print("HBD:", Descriptors.NumHDonors(m))

TPSA: 111.13000000000001
HBD: 4


In [ ]:
compounds = {
"Kaempferol": "C1=CC(=CC=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O",
"2-OH-cinnamic": "C1=CC=C(C(=C1)/C=C/C(=O)O)O",
"Rhapontigenin": "COC1=C(C=C(C=C1)/C=C/C2=CC(=CC(=C2)O)O)O",
"Ombuin": "COC1=C(C=C(C=C1)C2=C(C(=O)C3=C(C=C(C=C3O2)OC)O)O)O",
}
df2 = pd.DataFrame({"compound": list(compounds.keys()), "smiles": list(compounds.values())})
df2["mol"] = df2["smiles"].apply(Chem.MolFromSmiles)
df2["TPSA"] = df2["mol"].apply(Descriptors.TPSA).round(1)
df2["HBD"] = df2["mol"].apply(Descriptors.NumHDonors)
df2["P_brain"] = df2["mol"].apply(lambda mm: round(clf.predict_proba([[int(b) for b in
AllChem.GetMorganFingerprintAsBitVect(mm,2,nBits=2048).ToBitString()]])[0][1], 2))
df2[["compound","TPSA","HBD","P_brain"]]

[19:20:50] DEPRECATION WARNING: please use MorganGenerator
[19:20:50] DEPRECATION WARNING: please use MorganGenerator
[19:20:50] DEPRECATION WARNING: please use MorganGenerator
[19:20:50] DEPRECATION WARNING: please use MorganGenerator


,compound,TPSA,HBD,P_brain
0,Kaempferol,111.1,4,0.49
1,2-OH-cinnamic,57.5,2,0.82
2,Rhapontigenin,69.9,3,0.84
3,Ombuin,109.4,3,0.70
